In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-25 20:19:58--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.2.33, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  67.6MB/s    in 0.4s    

2026-05-25 20:19:58 (67.6 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [6]:
# add your code here - consider creating a new cell for each section of code

# remove non statistically significant entries
user_ratings = df_ratings["user"].value_counts()
book_ratings = df_ratings["isbn"].value_counts()

users_to_remove = []
books_to_remove = []

for user, val in user_ratings.items():
  if (val < 200):
    users_to_remove.append(val)

for book, val in book_ratings.items():
  if (val < 100):
    books_to_remove.append(val)

# had to figure out how to filter out by a list of undesired values, AI gave me this suggestion which worked
# I understand how this works but it just seems so weird
df_ratings = df_ratings[
    (~df_ratings["user"].isin(users_to_remove)) &
    (~df_ratings["isbn"].isin(books_to_remove))
]

In [7]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):

  matrix = csr_matrix(df_ratings.values)
  model = NearestNeighbors(
      metric='cosine',
      algorithm='brute'
  )

  model.fit(matrix)

  distances, indices = model.kneighbors(
    df_ratings.loc[[book]],
    n_neighbors=5
  )

  recommended_books = [book,[]]
  for i in range(5):
    # get isbn using index and find name
    isbn = df_ratings.at(indices[i])['isbn']
    recommended_books[1].append([df_books.at((df_books['isbn'] == isbn).idxmax())["title"], distances[i]])

  return recommended_books

In [8]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

ValueError: scipy.sparse does not support dtype object. The only supported types are: bool, int8, uint8, int16, uint16, int32, uint32, int64, uint64, longlong, ulonglong, float32, float64, longdouble, complex64, complex128, clongdouble.